
# PAWS-X (Spanish) Paraphrase Detection with XLM-RoBERTa

This notebook fine-tunes **XLM-R (xlm-roberta-base)** on the **PAWS-X Spanish** paraphrase dataset.
It is **version-robust** against older Transformers builds and includes short intros before each step.



**Cell purpose:** (Optional) Install/upgrade libraries if you are on a fresh environment.  
If you have version conflicts (e.g., with cuDF/RAPIDS), you may skip or adapt the pins.


In [1]:

# Optional: install/upgrade libs. Comment out if your env is already set.
!pip -q install -U transformers datasets evaluate accelerate sentencepiece
# If you use cuDF/RAPIDS and get a pyarrow conflict, consider pinning:
# !pip -q install "pyarrow<20"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.



**Cell purpose:** Import libraries, print versions, select device (GPU/CPU), and set random seeds.


In [ ]:

import os, random, numpy as np, torch, inspect, datasets, transformers, evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

print("Versions -> transformers:", transformers.__version__, "| datasets:", datasets.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)



**Cell purpose:** Load the **PAWS-X** dataset for Spanish.  
It contains pairs of sentences (`sentence1`, `sentence2`) and a binary `label` (0 = not paraphrase, 1 = paraphrase).


In [ ]:

dataset = load_dataset("paws-x", "es")
print(dataset)
print("Sample:", dataset["train"][0])
id2label = {0: "not-paraphrase", 1: "paraphrase"}
label2id = {"not-paraphrase": 0, "paraphrase": 1}



**Cell purpose:** Prepare the XLM-R tokenizer and map the dataset into token IDs for sentence pairs.  
We remove non-tensor columns dynamically (`id`/`idx`, `sentence1`, `sentence2`) to avoid schema issues.


In [ ]:

checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)

max_length = 256
def preprocess(batch):
    return tokenizer(batch["sentence1"], batch["sentence2"],
                     padding="max_length", truncation=True, max_length=max_length)

to_maybe_remove = ["idx", "id", "sentence1", "sentence2"]
def cols_present(split):
    return [c for c in to_maybe_remove if c in dataset[split].column_names]

encoded = {}
for split in ["train","validation","test"]:
    encoded[split] = dataset[split].map(preprocess, batched=True, remove_columns=cols_present(split))
    encoded[split] = encoded[split].rename_column("label", "labels")
    encoded[split].set_format(type="torch")

train_ds, eval_ds, test_ds = encoded["train"], encoded["validation"], encoded["test"]
len(train_ds), len(eval_ds), len(test_ds)



**Cell purpose:** Load **XLM-R** for sequence classification (2 labels) and move it to the selected device.


In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2, id2label=id2label, label2id=label2id
)
model.to(device);



**Cell purpose:** Define evaluation metrics (accuracy, macro precision/recall/F1) using `evaluate`.


In [ ]:

metric_accuracy  = evaluate.load("accuracy")
metric_precision = evaluate.load("precision")
metric_recall    = evaluate.load("recall")
metric_f1        = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": metric_accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision_macro": metric_precision.compute(predictions=preds, references=labels, average="macro")["precision"],
        "recall_macro": metric_recall.compute(predictions=preds, references=labels, average="macro")["recall"],
        "f1_macro": metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }



**Cell purpose:** Build `TrainingArguments` that adapt to your Transformers version.  
If `evaluation_strategy` and `save_strategy` are supported, we set both to `"epoch"` and enable best-model loading at the end.  
Otherwise, we use `eval_steps`/`save_steps` if available and avoid mismatched strategies.


In [ ]:

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
batch_size = 16

ta_params = inspect.signature(TrainingArguments.__init__).parameters
kwargs = dict(
    output_dir="xlmr-pawsx-es",
    logging_steps=100,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
)

# Add optional args if supported by your version
if "warmup_ratio" in ta_params: kwargs["warmup_ratio"] = 0.06
if "report_to" in ta_params: kwargs["report_to"] = "none"

supports_eval = "evaluation_strategy" in ta_params
supports_save = "save_strategy" in ta_params
supports_load_best = "load_best_model_at_end" in ta_params
supports_metric_for_best = "metric_for_best_model" in ta_params
supports_greater_is_better = "greater_is_better" in ta_params

if supports_eval and supports_save:
    # Modern API: match strategies and optionally enable best-model-at-end
    kwargs["evaluation_strategy"] = "epoch"
    kwargs["save_strategy"] = "epoch"
    if supports_load_best: kwargs["load_best_model_at_end"] = True
    if supports_metric_for_best: kwargs["metric_for_best_model"] = "accuracy"
    if supports_greater_is_better: kwargs["greater_is_better"] = True
else:
    # Legacy API: avoid strategy mismatch; use steps if available
    if "eval_steps" in ta_params: kwargs["eval_steps"] = 500
    if "save_steps" in ta_params: kwargs["save_steps"] = 500
    for k in ("load_best_model_at_end", "metric_for_best_model", "greater_is_better",
              "save_strategy", "evaluation_strategy"):
        kwargs.pop(k, None)

args = TrainingArguments(**kwargs)
args



**Cell purpose:** Create the `Trainer` with our datasets, collator, and metrics, then fine-tune the model.


In [ ]:

trainer_extra = {}
# Some very old Trainer versions accepted evaluate_during_training — add only if present
if "evaluate_during_training" in inspect.signature(Trainer.__init__).parameters:
    trainer_extra["evaluate_during_training"] = not (supports_eval and supports_save)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    **trainer_extra
)

trainer.train();



**Cell purpose:** Evaluate the fine-tuned model on the **test** split and print metrics.


In [ ]:

test_metrics = trainer.evaluate(test_ds)
test_metrics



**Cell purpose:** Define a convenience function to score whether two sentences are paraphrases.


In [ ]:

import torch.nn.functional as F

def predict_paraphrase(sentence1: str, sentence2: str):
    model.eval()
    with torch.no_grad():
        enc = tokenizer(sentence1, sentence2, return_tensors="pt",
                        truncation=True, padding=True, max_length=256).to(device)
        out = model(**enc)
        probs = F.softmax(out.logits, dim=-1).detach().cpu().numpy()[0]
        pred  = int(np.argmax(probs))
        return {
            "sentence1": sentence1,
            "sentence2": sentence2,
            "prediction": id2label[pred],
            "score": float(probs[pred]),
        }



**Cell purpose:** Run **five** Spanish examples (the model was trained on Spanish) — mix of paraphrase and non-paraphrase cases.


In [ ]:

examples = [
    # 1) Given example (likely paraphrase)
    ("El perro persigue al gato por el jardín.",
     "En el jardín, el perro va detrás del gato."),

    # 2) Clear paraphrase
    ("Madrid es la capital de España.",
     "La capital de España es Madrid."),

    # 3) Not paraphrase (landed vs took off)
    ("El avión aterrizó a las 7.",
     "El avión despegó a las 7."),

    # 4) Likely paraphrase (preference phrasing)
    ("Me gusta el café sin azúcar.",
     "Prefiero el café sin azúcar."),

    # 5) Not paraphrase (bought vs sold)
    ("Juan compró un coche rojo.",
     "Juan vendió un coche rojo."),
]

results = [predict_paraphrase(a, b) for a, b in examples]
for i, r in enumerate(results, 1):
    print(f"Example {i}: {r['prediction']} (score={r['score']:.3f})")
    print("  s1:", r["sentence1"])
    print("  s2:", r["sentence2"])
    print("-"*60)
